# Lab 3
Vladyslav Sydorak, Nazarii Dizhak

Побудувати систему роспізнавання мовлення на Toronto Dataset: 
1.  Обрати доцільний притрейн Whisper з huggingface  
2.  Імплементувати логіку тренування використовуючи Pytorch Lightning 
3.  Донавчіть модель Whisper на частині датасету Toronto. Також ви можете 
використовувати інші українські датасети для Speech-to-Text (або 
Text-to-Speech), наприклад, багато з них можна знайти тут 
4.  Не використовуйте наступну частину датасету Toronto ні для навчання, ні для валідації. Використовуйте її лише для тестування вашої фінальної моделі.  Використовуйте метрики CER та WER (а також інші, за бажанням).

```python
test_lines = ['toronto_27', 'toronto_46', 'toronto_42', 'toronto_37', 'toronto_89', 
'toronto_43', 'toronto_157', 'toronto_9', 'toronto_156', 'toronto_7', 
'toronto_123', 'toronto_54', 'toronto_67', 'toronto_62', 'toronto_81', 
'toronto_134', 'toronto_148', 'toronto_21', 'toronto_135', 'toronto_166', 
'toronto_58'] 
```

In [ ]:
import os
import re
import json
import torch
import torchaudio
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor, TQDMProgressBar
from torch.utils.data import Dataset, DataLoader
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from torchmetrics.text import WordErrorRate, CharErrorRate
from typing import Any, Dict, List, Union

In [2]:
MODEL_NAME = "openai/whisper-small"
JSON_LABELS_PATH = "/kaggle/input/datasets/vladyslavsydorak/toronto/dataset/labels.jsonl"
BATCH_SIZE = 8

test_lines = [
    'toronto_27', 'toronto_46', 'toronto_42', 'toronto_37', 'toronto_89',
    'toronto_43', 'toronto_157', 'toronto_9', 'toronto_156', 'toronto_7',
    'toronto_123', 'toronto_54', 'toronto_67', 'toronto_62', 'toronto_81',
    'toronto_134', 'toronto_148', 'toronto_21', 'toronto_135', 'toronto_166',
    'toronto_58'
]

In [3]:
class TorontoDataset(Dataset):
    def __init__(self, json_path: str, processor: WhisperProcessor, split: str = 'train', test_lines: List[str] = None):
        self.processor = processor
        self.sampling_rate = processor.feature_extractor.sampling_rate

        base_dir = "/kaggle/input/datasets/vladyslavsydorak/toronto"

        with open(json_path, 'r', encoding='utf-8') as f:
            data_dict = json.load(f)

        test_lines = test_lines or []
        self.data = []
        missing_files_count = 0

        for relative_path, transcript in data_dict.items():
            full_path = os.path.join(base_dir, relative_path)

            if not os.path.exists(full_path):
                missing_files_count += 1
                continue

            is_test = any(test_id in full_path for test_id in test_lines)
            
            if split == 'test' and is_test:
                self.data.append({"path": full_path, "text": transcript})
            elif split in ['train', 'val'] and not is_test:
                is_val = hash(full_path) % 10 == 0
                if (split == 'val' and is_val) or (split == 'train' and not is_val):
                    self.data.append({"path": full_path, "text": transcript})

        if missing_files_count > 0:
            print(f"[{split.upper()}] Skipped {missing_files_count} missing files.")
            if self.data:
                print(f"Sample path found: {self.data[0]['path']}")
            else:
                print(f"Attempted to find files at: {os.path.join(base_dir, list(data_dict.keys())[0])}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        waveform, sr = torchaudio.load(item["path"])
        
        if sr != self.sampling_rate:
            waveform = torchaudio.functional.resample(waveform, orig_freq=sr, new_freq=self.sampling_rate)

        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        input_features = self.processor.feature_extractor(
            waveform.squeeze().numpy(), 
            sampling_rate=self.sampling_rate, 
            return_tensors="pt"
        ).input_features[0]

        labels = self.processor.tokenizer(item["text"], return_tensors="pt").input_ids[0]
        return {"input_features": input_features, "labels": labels}

In [4]:
class DataCollatorSpeechSeq2SeqWithPadding:
    def __init__(self, processor: Any):
        self.processor = processor

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

In [5]:
class WhisperPLModule(pl.LightningModule):
    def __init__(self, model_name_or_path: str, learning_rate: float = 1e-5):
        super().__init__()
        self.save_hyperparameters()
        self.learning_rate = learning_rate

        self.processor = WhisperProcessor.from_pretrained(
            model_name_or_path, language="uk", task="transcribe"
        )
        self.model = WhisperForConditionalGeneration.from_pretrained(model_name_or_path)

        self.model.train()

        self.model.generation_config.language = "uk"
        self.model.generation_config.task = "transcribe"
        self.model.generation_config.forced_decoder_ids = None

        self.wer_metric = WordErrorRate()
        self.cer_metric = CharErrorRate()

    def forward(self, input_features, labels=None):
        return self.model(input_features=input_features, labels=labels)

    def training_step(self, batch, batch_idx):
        outputs = self(input_features=batch["input_features"], labels=batch["labels"])
        loss = outputs.loss
        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss, preds_str, labels_str = self._shared_eval_step(batch)
        self.log("val_loss", loss, prog_bar=True, sync_dist=True)

        self.wer_metric.update(preds_str, labels_str)
        self.cer_metric.update(preds_str, labels_str)
        return loss

    def on_validation_epoch_end(self):
        self.log("val_wer", self.wer_metric.compute(), prog_bar=True)
        self.log("val_cer", self.cer_metric.compute(), prog_bar=True)
        self.wer_metric.reset()
        self.cer_metric.reset()

    def test_step(self, batch, batch_idx):
        loss, preds_str, labels_str = self._shared_eval_step(batch)
        self.wer_metric.update(preds_str, labels_str)
        self.cer_metric.update(preds_str, labels_str)
        return loss

    def on_test_epoch_end(self):
        wer = self.wer_metric.compute()
        cer = self.cer_metric.compute()
        self.log("test_wer", wer)
        self.log("test_cer", cer)
        print(f"\nFinal Test Results -> WER: {wer:.4f} | CER: {cer:.4f}")
        self.wer_metric.reset()
        self.cer_metric.reset()

    def _normalize_text(self, text: str) -> str:
        text = text.lower()
        text = re.sub(r'[^\w\s]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def _shared_eval_step(self, batch):
        input_features = batch["input_features"]
        labels = batch["labels"]

        outputs = self(input_features=input_features, labels=labels)
        loss = outputs.loss

        attention_mask = torch.ones(
            input_features.shape[0],
            input_features.shape[2],
            dtype=torch.long,
            device=input_features.device
        )

        pred_ids = self.model.generate(
            input_features,
            attention_mask=attention_mask
        )

        labels[labels == -100] = self.processor.tokenizer.pad_token_id

        preds_str = self.processor.batch_decode(pred_ids, skip_special_tokens=True)
        labels_str = self.processor.batch_decode(labels, skip_special_tokens=True)

        preds_str = [self._normalize_text(text) for text in preds_str]
        labels_str = [self._normalize_text(text) for text in labels_str]

        return loss, preds_str, labels_str

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=self.learning_rate, weight_decay=0.01)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=1000)
        return {"optimizer": optimizer, "lr_scheduler": scheduler}

In [6]:
model = WhisperPLModule(model_name_or_path=MODEL_NAME)
processor = model.processor

preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

In [7]:
train_dataset = TorontoDataset(JSON_LABELS_PATH, processor, split='train', test_lines=test_lines)
val_dataset = TorontoDataset(JSON_LABELS_PATH, processor, split='val', test_lines=test_lines)
test_dataset = TorontoDataset(JSON_LABELS_PATH, processor, split='test', test_lines=test_lines)

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=data_collator, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator, num_workers=2)

[TRAIN] Skipped 10929 missing files.
Sample path found: /kaggle/input/datasets/vladyslavsydorak/toronto/dataset/toronto_145/toronto_145_1.wav
[VAL] Skipped 10929 missing files.
Sample path found: /kaggle/input/datasets/vladyslavsydorak/toronto/dataset/toronto_145/toronto_145_0.wav
[TEST] Skipped 10929 missing files.
Sample path found: /kaggle/input/datasets/vladyslavsydorak/toronto/dataset/toronto_157/toronto_157_0.wav


In [8]:
torch.cuda.is_available()

True

In [9]:
callbacks = [
    TQDMProgressBar(refresh_rate=10),
    LearningRateMonitor(logging_interval='step'),
    ModelCheckpoint(
        dirpath="checkpoints",
        filename="whisper-uk-{epoch:02d}-{val_wer:.2f}",
        monitor="val_wer",
        mode="min",
        save_top_k=1
    )
]

trainer = pl.Trainer(
    max_epochs=10,
    accelerator="gpu",
    devices=1,
    precision="16-mixed",
    gradient_clip_val=1.0,
    callbacks=callbacks,
    log_every_n_steps=10
)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [10]:
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

2026-04-22 19:53:31.906797: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776887612.231702      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776887612.338674      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776887613.207580      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776887613.207632      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776887613.207635      55 computation_placer.cc:177] computation placer alr

┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model      │ WhisperForConditionalGeneration │  241 M │ train │     0 │
│ 1 │ wer_metric │ WordErrorRate                   │      0 │ train │     0 │
│ 2 │ cer_metric │ CharErrorRate                   │      0 │ train │     0 │
└───┴────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 241 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 241 M                                                                                                
Total estimated model params size (MB): 966                                                                        
Modules in train mode: 352                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will 

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=10` reached.


In [11]:
trainer.test(model, dataloaders=test_loader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Testing: |          | 0/? [00:00<?, ?it/s]


Final Test Results -> WER: 0.2565 | CER: 0.1107


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_cer          │    0.11069232225418091    │
│         test_wer          │    0.2565237283706665     │
└───────────────────────────┴───────────────────────────┘

[{'test_wer': 0.2565237283706665, 'test_cer': 0.11069232225418091}]